# Non-Motorized Sensitivity Test — SE File Prep

Generates one modified SE CSV per scenario (S01–S13) into `../inputs/`.  
Run this notebook once before executing the run set with `tdmruns run-set --run-set non-motorized-2026`.

**Spatial scope legend**
- `smldst` — small districts listed in `small_district_ids` (TAZs resolved via shapefile lookup)
- `smldst+taz` — small districts PLUS the specific TAZs in `taz_ids`
- `region` — all TAZs (whole model area)

In [ ]:
from pathlib import Path

taz_shapefile = r"..\..\tdm\1_Inputs\1_TAZ\WFv910_TAZ.shp"
se_input_dir = r"..\..\tdm\1_Inputs\2_SEData\2_WFRC\FiscallyConstrained"

# Output goes into this run_set's inputs/ folder (relative to this notebook).
output_dir = "../inputs"
print(f"Output directory: {output_dir}")


Output directory: ../inputs


In [ ]:
import pandas as pd
import geopandas as gpd
import os


In [ ]:
taz_gdf = gpd.read_file(taz_shapefile)
taz_gdf.rename(columns={"TAZID": ";TAZID"}, inplace=True)
taz_smldst_lookup_df = taz_gdf[[";TAZID", "DISTSML"]]
display(taz_smldst_lookup_df)


,;TAZID,DISTSML
0,1,1
1,2,1
2,3,1
3,4,1
4,5,1
...,...,...
3541,3506,120
3542,3508,120
3543,3504,120
3544,3546,126


In [ ]:
field_mapping_df = pd.DataFrame(
    [
        ["HH", ["TOTHH", "HHPOP"]],
        [
            "EMP",
            [
                "TOTEMP",
                "RETEMP",
                "OTHEMP",
                "INDEMP",
                "FOOD",
                "RETL",
                "WSLE",
                "MANU",
                "OFFI",
                "GVED",
                "HLTH",
                "OTHR",
            ],
        ],
    ],
    columns=["se_field_name", "all_applicable_fieldnames"],
)

field_mapping_exploded = field_mapping_df.explode("all_applicable_fieldnames")
field_mapping_exploded


,se_field_name,all_applicable_fieldnames
0,HH,TOTHH
0,HH,HHPOP
1,EMP,TOTEMP
1,EMP,RETEMP
1,EMP,OTHEMP
1,EMP,INDEMP
1,EMP,FOOD
1,EMP,RETL
1,EMP,WSLE
1,EMP,MANU


## Scenario definitions

Each row defines one scenario. The `test_id` maps to the scenario ID (S01–S13).  
- `smldst=1` applies multipliers to TAZs in the small districts listed below  
- `taz=1` additionally applies to the specific TAZs listed below  
- `region=1` applies to all TAZs  
- Empty `multiplier_params` means copy the base file as-is (file substitution only)

In [ ]:
# Spatial scope for this study
taz_lst = [1643, 2030, 2741, 2834]
smldst_lst = [22, 16, 46, 88, 102]


In [ ]:
# fmt: off
test_localized_df = pd.DataFrame(
    [
        [1, "S01", "2019 HH x2"                     , "SE_2019.csv"                  , [["HH", 2]]              , 1, 0, 0],
        [2, "S02", "2019 HH x4"                     , "SE_2019.csv"                  , [["HH", 4]]              , 1, 1, 0],
        [3, "S03", "2019 HH x12"                    , "SE_2019.csv"                  , [["HH", 12]]             , 1, 1, 0],
        [4, "S04", "2019 EMP x2"                    , "SE_2019.csv"                  , [["EMP", 2]]             , 1, 0, 0],
        [5, "S05", "2019 EMP x4"                    , "SE_2019.csv"                  , [["EMP", 4]]             , 1, 1, 0],
        [6, "S06", "2019 EMP x12"                   , "SE_2019.csv"                  , [["EMP", 12]]            , 1, 1, 0],
        [7, "S07", "2019 HH x2 and EMP x2"          , "SE_2019.csv"                  , [["HH", 2], ["EMP", 2]]  , 1, 0, 0],
        [8, "S08", "2019 HH x4 and EMP x4"          , "SE_2019.csv"                  , [["HH", 4], ["EMP", 4]]  , 1, 1, 0],
        [9, "S09", "2019 HH x12 and EMP x12"        , "SE_2019.csv"                  , [["HH", 12], ["EMP", 12]], 1, 1, 0],
        [10, "S10", "Future Today SE_2050"          , "SE_2050.csv"                  , []                       , 1, 0, 0],
        [11, "S11", "Future Today Centerized"       , "SE_2050_transit_corridors.csv", []                       , 1, 0, 0],
        [12, "S12", "Region Future Today SE_2050"   , "SE_2050.csv"                  , []                       , 0, 0, 1],
        [13, "S13", "Region Future Today Centerized", "SE_2050_transit_corridors.csv", []                       , 0, 0, 1]
    ],
    columns=[
        "test_id", "scenario_id", "description", "replacement_se_file", "multiplier_params", "smldst", "taz", "region",
    ],
)
# fmt: on

test_localized_multiplier_df = test_localized_df.explode("multiplier_params")
test_localized_multiplier_df["se_field_name"] = test_localized_multiplier_df[
    "multiplier_params"
].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
test_localized_multiplier_df["multiplier"] = test_localized_multiplier_df[
    "multiplier_params"
].apply(lambda x: x[1] if isinstance(x, list) and len(x) > 1 else None)
test_localized_multiplier_df = pd.merge(
    test_localized_multiplier_df, field_mapping_exploded, on="se_field_name", how="left"
)
display(test_localized_multiplier_df)


,test_id,scenario_id,description,replacement_se_file,multiplier_params,smldst,taz,region,se_field_name,multiplier,all_applicable_fieldnames
0,1,S01,2019 HH x2,SE_2019.csv,"[HH, 2]",1,0,0,HH,2.0,TOTHH
1,1,S01,2019 HH x2,SE_2019.csv,"[HH, 2]",1,0,0,HH,2.0,HHPOP
2,2,S02,2019 HH x4,SE_2019.csv,"[HH, 4]",1,1,0,HH,4.0,TOTHH
3,2,S02,2019 HH x4,SE_2019.csv,"[HH, 4]",1,1,0,HH,4.0,HHPOP
4,3,S03,2019 HH x12,SE_2019.csv,"[HH, 12]",1,1,0,HH,12.0,TOTHH
...,...,...,...,...,...,...,...,...,...,...,...
83,9,S09,2019 HH x12 and EMP x12,SE_2019.csv,"[EMP, 12]",1,1,0,EMP,12.0,OTHR
84,10,S10,Future Today SE_2050,SE_2050.csv,NaN,1,0,0,None,NaN,NaN
85,11,S11,Future Today Centerized,SE_2050_transit_corridors.csv,NaN,1,0,0,None,NaN,NaN
86,12,S12,Region Future Today SE_2050,SE_2050.csv,NaN,0,0,1,None,NaN,NaN


In [ ]:
se_file_lst = test_localized_df["replacement_se_file"].unique().tolist()
se_data_dfs = {f: pd.read_csv(os.path.join(se_input_dir, f)) for f in se_file_lst}


In [ ]:
localized_se_results = {}

for _, scenario_row in test_localized_df.iterrows():
    test_id = scenario_row["test_id"]
    se_file = scenario_row["replacement_se_file"]
    df = se_data_dfs[se_file].copy()
    df["modified"] = 0

    test_rows = test_localized_multiplier_df[test_localized_multiplier_df["test_id"] == test_id]
    for _, row in test_rows.iterrows():
        if pd.notnull(row["multiplier"]) and pd.notnull(row["all_applicable_fieldnames"]):
            field = row["all_applicable_fieldnames"]
            multiplier = row["multiplier"]
            if row["taz"] == 1:
                df.loc[df[";TAZID"].isin(taz_lst), field] *= multiplier
                df.loc[df[";TAZID"].isin(taz_lst), "modified"] = 1
            if row["smldst"] == 1:
                smldst_tazs = taz_smldst_lookup_df[
                    taz_smldst_lookup_df["DISTSML"].isin(smldst_lst)
                ][";TAZID"].tolist()
                df.loc[df[";TAZID"].isin(smldst_tazs), field] *= multiplier
                df.loc[df[";TAZID"].isin(smldst_tazs), "modified"] = 1
            if row["region"] == 1:
                df[field] *= multiplier
                df["modified"] = 1

    localized_se_results[scenario_row["scenario_id"]] = df

print(f"Generated SE frames for: {list(localized_se_results.keys())}")


Generated SE frames for: ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13']


In [ ]:
for scenario_id, df_out in localized_se_results.items():
    # Drop the helper column before saving
    out_df = df_out.drop(columns=["modified"])
    out_path = os.path.join(output_dir, f"SE_{scenario_id}.csv")
    out_df.to_csv(out_path, index=False)
    print(f"Wrote {out_path}  ({len(out_df):,} rows)")


Wrote ../inputs\SE_S01.csv  (3,546 rows)
Wrote ../inputs\SE_S02.csv  (3,546 rows)
Wrote ../inputs\SE_S03.csv  (3,546 rows)
Wrote ../inputs\SE_S04.csv  (3,546 rows)
Wrote ../inputs\SE_S05.csv  (3,546 rows)
Wrote ../inputs\SE_S06.csv  (3,546 rows)
Wrote ../inputs\SE_S07.csv  (3,546 rows)
Wrote ../inputs\SE_S08.csv  (3,546 rows)
Wrote ../inputs\SE_S09.csv  (3,546 rows)
Wrote ../inputs\SE_S10.csv  (3,546 rows)
Wrote ../inputs\SE_S11.csv  (3,546 rows)
Wrote ../inputs\SE_S12.csv  (3,546 rows)
Wrote ../inputs\SE_S13.csv  (3,546 rows)
